In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
bronze_table = f"{catalog}.{schema}.bronze_orders"
silver_table = f"{catalog}.{schema}.silver_orders"

bronze_df = spark.table(bronze_table)

silver_df = (
    bronze_df
    .select(
        F.trim(F.col("order_id")).cast("long").alias("order_id"),
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.to_date(F.col("order_date"), "yyyy-MM-dd").alias("order_date"),
        F.trim(F.col("product")).alias("product"),
        F.initcap(F.trim(F.col("category"))).alias("category"),
        F.col("quantity").cast("int").alias("quantity"),
        F.col("unit_price").cast("decimal(12,2)").alias("unit_price"),
        F.initcap(F.trim(F.col("city"))).alias("city"),
        F.lower(F.trim(F.col("status"))).alias("status")
    )
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id") != ""))
    .filter(F.col("order_date").isNotNull())
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") > 0)
    .dropDuplicates(["order_id"])
    .withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
)

(silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table))

print(f"Silver cleaning complete: {silver_table}")
display(spark.table(silver_table).orderBy("order_id"))